# XGBoost Classification of Plant Alliances

This notebook trains an XGBoost classifier to predict phytosociological alliances from vegetation relevé vectors using Optuna for hyperparameter optimization.

## 1. Load Dependencies

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
import xgboost as xgb
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, log_loss

## 2. Load and Prepare Data

In [ ]:
# Load vectorized relev? data
file_path = 'Matriu_de_dades/output_pirineus_3_corrected.txt'
with open(file_path, 'r') as file:
    loaded_vector_dic = json.load(file)

# Build feature matrix
df = pd.DataFrame.from_dict(loaded_vector_dic, orient='index', columns=['List', 'Label'])
df = pd.DataFrame(df['List'].tolist(), index=df.index)
df.columns = [f'Column_{i+1}' for i in range(len(df.columns))]

# Add target labels
labels = [loaded_vector_dic[idx][1] for idx in df.index]
df['Target'] = labels

# Filter classes with fewer than 21 samples
MIN_SAMPLES = 21
label_counts = df['Target'].value_counts()
valid_labels = label_counts[label_counts >= MIN_SAMPLES].index.tolist()
df_filtered = df[df['Target'].isin(valid_labels)]

print(f"Dataset shape: {df_filtered.shape}")
print(f"Number of classes: {len(valid_labels)}")

## 3. Encode Labels and Split Data

In [ ]:
label_encoder = LabelEncoder()

X = df_filtered.drop(columns='Target')
y = label_encoder.fit_transform(df_filtered['Target'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=8
)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 4. Hyperparameter Optimization with Optuna

In [ ]:
def objective(trial):
    param = {
        'objective': 'multi:softprob',
        'eval_metric': 'mlogloss',
        'booster': 'gbtree',
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-4, 1e-1),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_loguniform('gamma', 1e-8, 1.0),
        'subsample': trial.suggest_uniform('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.7, 1.0),
        'lambda': trial.suggest_loguniform('lambda', 1e-8, 10.0),
        'alpha': trial.suggest_loguniform('alpha', 1e-8, 10.0),
        'num_class': len(set(y_train)),
        'seed': 8
    }

    n_estimators = trial.suggest_int('n_estimators', 100, 800)

    dtrain = xgb.DMatrix(X, label=y)
    cv_results = xgb.cv(
        param, dtrain,
        num_boost_round=n_estimators,
        nfold=5,
        stratified=True,
        metrics='mlogloss',
        early_stopping_rounds=10,
        seed=8
    )

    return cv_results['test-mlogloss-mean'].min()

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, n_jobs=10)
best_params = study.best_params
print(f"Best parameters: {best_params}")

## 5. Train Model with Best Parameters

In [ ]:
model = xgb.XGBClassifier(**best_params)
model.fit(
    X_train, y_train,
    eval_metric='mlogloss',
    eval_set=[(X_train, y_train), (X_test, y_test)],
    early_stopping_rounds=10,
    verbose=False
)

train_preds = model.predict(X_train)
test_preds = model.predict(X_test)

print(f"Training Accuracy: {accuracy_score(y_train, train_preds):.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")

## 6. Learning Curves

In [ ]:
evals_result = model.evals_result()
num_rounds = len(evals_result['validation_0']['mlogloss'])

plt.figure(figsize=(10, 5))
plt.plot(range(1, num_rounds + 1), evals_result['validation_0']['mlogloss'], label='Train Log Loss')
plt.plot(range(1, num_rounds + 1), evals_result['validation_1']['mlogloss'], label='Validation Log Loss')
plt.xlabel('Boosting Rounds')
plt.ylabel('Log Loss')
plt.title('XGBoost Learning Curves')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Cross-Validation

In [ ]:
model_cv = xgb.XGBClassifier(**best_params)
skf = StratifiedKFold(n_splits=5)
cv_scores = cross_val_score(model_cv, X, y, cv=skf, scoring='accuracy')

print(f"Cross-validated accuracy per fold: {cv_scores}")
print(f"Mean CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 8. Classification Report

In [ ]:
class_names = label_encoder.inverse_transform(np.unique(y_test))
test_preds = model.predict(X_test)

report = classification_report(y_test, test_preds, target_names=class_names,
                               zero_division=0, output_dict=True)
report_df = pd.DataFrame(report).transpose()
print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
report_df.to_excel('cls_report_test_xgb.xlsx')
report_df.head(10)

## 9. Detailed Predictions with Top-3 Probabilities

In [ ]:
def get_top_3_predictions(proba_vector, label_mapping):
    """Get top 3 predicted classes and their probabilities."""
    top_3_indices = np.argsort(proba_vector)[-3:][::-1]
    top_3_proba = proba_vector[top_3_indices]
    top_3_labels = [label_mapping[idx] for idx in top_3_indices]
    return top_3_labels, top_3_proba

# Recover original relev? IDs using the same random state
X_train_a, X_test_a, _, _ = train_test_split(X, y, test_size=0.3, stratify=y, random_state=8)
label_mapping = dict(enumerate(label_encoder.classes_))

# Generate predictions
prob_test = model.predict_proba(X_test)
top_3_test = [get_top_3_predictions(p, label_mapping) for p in prob_test]

df_results = pd.DataFrame({
    'releve_id': X_test_a.index,
    'prediction': label_encoder.inverse_transform(model.predict(X_test)),
    'true_label': label_encoder.inverse_transform(y_test),
    'top_3_labels': [t[0] for t in top_3_test],
    'top_3_probabilities': [t[1] for t in top_3_test]
})

df_results.to_excel('predictions_test_xgb.xlsx', index=False)
df_results.head()

## 10. Train Final Model and Save

In [ ]:
# Train on full dataset for deployment
model_final = xgb.XGBClassifier(**best_params)
model_final.fit(X, y)

joblib.dump(model_final, 'models/xgb_final_model.pkl')
print("Final model saved to models/xgb_final_model.pkl")

## 11. Inference on New Data

In [ ]:
# Load unlabeled relev?s
file_path_new = 'Matriu_de_dades/releves_sense_alianca_vectors.txt'
with open(file_path_new, 'r') as file:
    new_vector_dic = json.load(file)

df_new = pd.DataFrame.from_dict(new_vector_dic, orient='index', columns=['List', 'Label'])
X_new = pd.DataFrame(df_new['List'].tolist(), index=df_new.index)
X_new.columns = [f'Column_{i+1}' for i in range(len(X_new.columns))]

# Predict
model_loaded = joblib.load('models/xgb_final_model.pkl')
preds = model_loaded.predict(X_new)
preds_prob = model_loaded.predict_proba(X_new)

# Build output dataframe
predicted_labels = label_encoder.inverse_transform(preds)
top_3 = [get_top_3_predictions(p, label_mapping) for p in preds_prob]

df_output = pd.DataFrame({
    'Releve_id': X_new.index.tolist(),
    'Alliance': predicted_labels,
    'Top_3_alliances': [t[0] for t in top_3],
    'Top_3_probabilities': [t[1] for t in top_3]
})

df_output.to_excel('predictions_new_data_xgb.xlsx', index=False)
print(f"Predictions saved for {len(df_output)} relev?s")
df_output.head()